# Demo 5 — Curve train/validation e checkpoint

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

**Risultati registrati:** questa demo esplora il manifest e le predizioni dell’esperimento già eseguito. Non addestra la CNN. Le metriche e i grafici vengono ricalcolati nel browser; i clip degli errori sono scaricati solo su richiesta.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
epoche_visibili = 8               # da 1 a 8; modifica solo la visualizzazione
risultati = al.load_classification_results(DEMO_ASSETS / "classification-results.json")
info = risultati["cnn"]
storia = [r for r in info["history"] if r["epoch"]<=epoche_visibili]
assert storia, "Visualizzare almeno un'epoca."
epoche = [r["epoch"] for r in storia]
fig = make_subplots(rows=1,cols=2,subplot_titles=("Loss train","Macro-F1 train e validazione"))
fig.add_trace(go.Scatter(x=epoche,y=[r["loss"] for r in storia],name="Loss train"),row=1,col=1)
for campo,nome in (("train_f1","Train"),("val_f1","Validazione")):
    fig.add_trace(go.Scatter(x=epoche,y=[r[campo] for r in storia],name=nome),row=1,col=2)
if info["best_epoch"]<=epoche_visibili:
    fig.add_vline(x=info["best_epoch"],line_dash="dash",annotation_text="Checkpoint registrato",row=1,col=2)
fig.update_xaxes(title_text="Epoca",dtick=1)
fig.update_yaxes(range=[0,1],row=1,col=2)
fig.update_layout(title="Addestramento originale già concluso",height=400)
fig.show()
print("Epoca scelta sulla validazione:",info["best_epoch"])
print("Tempo di addestramento registrato:",round(info["seconds"],1),"s")
print("Checkpoint:",risultati["provenance"]["checkpoint"])
print("Queste curve sono quelle salvate durante l'addestramento; il browser non riaddestra la CNN.")
